# NLP YouTube Clickbait Classifier

A self-contained training and inference notebook for the text branch. It uses the existing `train.csv`, `validation.csv`, and `test.csv` files produced for the multimodal project; it never creates replacement splits. Each CSV needs `video_id`, `title`, `description`, and `label` (or `final_label`) columns.

The model is `distilbert/distilbert-base-cased`. Titles and descriptions are passed separately as a sentence pair. Descriptions have URLs replaced by `[URL]`; titles retain capitalization and punctuation. A title that exceeds the pair token budget raises an error with its video ID rather than being truncated.

Install project dependencies first. Set `SPLIT_DIR` below to the existing split directory. If another modality has a separate copy of the splits, set `REFERENCE_SPLIT_DIR` to that directory to check exact video IDs and labels.

In [1]:
from __future__ import annotations

import html
import json
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
SPLIT_DIR = ROOT / 'data' / 'processed'
REFERENCE_SPLIT_DIR = None  # e.g. ROOT / 'data' / 'cnn_splits'
OUTPUT_DIR = ROOT / 'models' / 'text_classifier'
MODEL_NAME = 'distilbert/distilbert-base-cased'
MAX_LENGTH = 256
SEED = 42
EPOCHS = 10
PATIENCE = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
URL_RE = re.compile(r'(?i)\b(?:https?://|www\.)\S+')

def clean_description(value):
    if value is None or pd.isna(value):
        return ''
    return URL_RE.sub('[URL]', str(value))

def read_split(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing existing split file: {path}. Provide the multimodal train/validation/test CSVs; this notebook will not create new splits.')
    frame = pd.read_csv(path, dtype={'video_id': 'string'})
    if 'label' not in frame.columns and 'final_label' in frame.columns:
        frame = frame.rename(columns={'final_label': 'label'})
    required = {'video_id', 'title', 'description', 'label'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{path} is missing columns: {sorted(missing)}')
    frame = frame.copy()
    frame['video_id'] = frame['video_id'].astype('string').str.strip()
    if frame.video_id.isna().any() or frame.video_id.eq('').any() or frame.video_id.duplicated().any():
        raise ValueError(f'{path} contains empty or duplicate video IDs')
    if frame.title.isna().any() or frame.title.astype(str).str.strip().eq('').any():
        raise ValueError(f'{path} contains missing or empty titles')
    labels = pd.to_numeric(frame.label, errors='coerce')
    if labels.isna().any() or not labels.isin([0, 1]).all():
        raise ValueError(f'{path} labels must be 0 or 1')
    frame['label'] = labels.astype('int64')
    frame['title'] = frame.title.astype(str)  # preserve original case and punctuation
    frame['description'] = frame.description.map(clean_description)
    return frame

train_df = read_split(SPLIT_DIR / 'train.csv')
validation_df = read_split(SPLIT_DIR / 'validation.csv')
test_df = read_split(SPLIT_DIR / 'test.csv')
splits = {'train': train_df, 'validation': validation_df, 'test': test_df}
for name, frame in splits.items():
    if frame.empty or set(frame.label.unique()) != {0, 1}:
        raise ValueError(f'{name} split must be nonempty and contain both labels')
ids = {name: set(frame.video_id.astype(str)) for name, frame in splits.items()}
for left, right in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    overlap = ids[left] & ids[right]
    if overlap:
        raise ValueError(f'Data leakage: {left} and {right} overlap, e.g. {sorted(overlap)[:5]}')
if REFERENCE_SPLIT_DIR is not None:
    for name, frame in splits.items():
        reference = read_split(Path(REFERENCE_SPLIT_DIR) / f'{name}.csv')
        assignments = dict(zip(frame.video_id.astype(str), frame.label))
        reference_assignments = dict(zip(reference.video_id.astype(str), reference.label))
        if assignments != reference_assignments:
            raise ValueError(f'{name} IDs or labels do not match the reference modality split')

print({name: {'rows': len(frame), 'label_counts': frame.label.value_counts().sort_index().to_dict()} for name, frame in splits.items()})

{'train': {'rows': 2957, 'label_counts': {0: 2332, 1: 625}}, 'validation': {'rows': 634, 'label_counts': {0: 500, 1: 134}}, 'test': {'rows': 634, 'label_counts': {0: 500, 1: 134}}}


In [2]:
class PairDataset(Dataset):
    def __init__(self, frame, tokenizer):
        self.rows = frame[['video_id', 'title', 'description', 'label']].to_dict('records')
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        title_tokens = self.tokenizer(row['title'], add_special_tokens=False, truncation=False)['input_ids']
        title_budget = MAX_LENGTH - self.tokenizer.num_special_tokens_to_add(pair=True)
        if len(title_tokens) > title_budget:
            raise ValueError(f'Title for video_id={row["video_id"]} needs {len(title_tokens)} tokens; pair budget permits {title_budget}. Review the title explicitly; no title tokens were dropped.')
        encoded = self.tokenizer(text=row['title'], text_pair=clean_description(row['description']),
                                 max_length=MAX_LENGTH, truncation='only_second',
                                 padding='max_length', return_attention_mask=True,
                                 return_tensors='pt')
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item['labels'] = torch.tensor(row['label'], dtype=torch.long)
        return item

class ClickbaitClassifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        layers = self.encoder.transformer.layer
        if len(layers) != 6:
            raise ValueError(f'Expected 6 DistilBERT layers, found {len(layers)}')
        for parameter in self.encoder.embeddings.parameters():
            parameter.requires_grad = False
        for layer in layers[:4]:
            for parameter in layer.parameters():
                parameter.requires_grad = False
        for layer in layers[4:]:
            for parameter in layer.parameters():
                parameter.requires_grad = True
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.encoder.config.dim, 2)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        logits = self.classifier(self.dropout(hidden[:, 0]))
        result = {'logits': logits}
        if labels is not None:
            result['loss'] = self.loss_fn(logits, labels)
        return result

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def metric_report(labels, probabilities):
    predictions = (np.asarray(probabilities) >= 0.5).astype(int)
    return {'accuracy': float(accuracy_score(labels, predictions)),
            'precision': float(precision_score(labels, predictions, zero_division=0)),
            'recall': float(recall_score(labels, predictions, zero_division=0)),
            'f1': float(f1_score(labels, predictions, zero_division=0)),
            'roc_auc': float(roc_auc_score(labels, probabilities)),
            'confusion_matrix': confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
            'predictions': predictions.tolist()}

@torch.inference_mode()
def evaluate(model, loader, device, amp_dtype=None):
    model.eval(); losses = []; labels = []; probabilities = []
    for batch in loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            output = model(**batch)
        losses.append(output['loss'].float().item())
        probabilities.extend(output['logits'].float().softmax(1)[:, 1].cpu().tolist())
        labels.extend(batch['labels'].cpu().tolist())
    report = metric_report(labels, probabilities)
    report.update({'loss': float(np.mean(losses)), 'labels': labels, 'probabilities': probabilities})
    return report

## Train and select with validation F1

The test loader is not evaluated during training. The best validation-F1 checkpoint is restored before the test metrics are calculated. CUDA and mixed precision are used where available.

In [3]:
seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 8 if device.type == 'cuda' else 2
amp_dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device.type == 'cuda' else None
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda' and amp_dtype == torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
datasets = {name: PairDataset(frame, tokenizer) for name, frame in splits.items()}
loaders = {name: DataLoader(dataset, batch_size=batch_size, shuffle=(name == 'train'),
                            pin_memory=device.type == 'cuda', num_workers=0)
           for name, dataset in datasets.items()}
model = ClickbaitClassifier().to(device)
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                              lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(loaders['train'])
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.1), total_steps)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
history = []; best_f1 = -1.0; best_epoch = 0; stale_epochs = 0
checkpoint_path = OUTPUT_DIR / 'best_model.pt'
for epoch in range(1, EPOCHS + 1):
    started = time.time(); model.train(); train_losses = []
    for batch in loaders['train']:
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            output = model(**batch)
        if scaler.is_enabled():
            scaler.scale(output['loss']).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            output['loss'].backward()
            torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
            optimizer.step()
        scheduler.step(); train_losses.append(output['loss'].detach().float().item())
    validation_report = evaluate(model, loaders['validation'], device, amp_dtype)
    entry = {'epoch': epoch, 'train_loss': float(np.mean(train_losses)),
             'validation': {k: v for k, v in validation_report.items() if k not in {'labels', 'probabilities', 'predictions'}},
             'seconds': round(time.time() - started, 2)}
    history.append(entry); print(entry)
    if validation_report['f1'] > best_f1:
        best_f1 = validation_report['f1']; best_epoch = epoch; stale_epochs = 0
        torch.save({'model_state_dict': model.state_dict(), 'model_name': MODEL_NAME,
                    'max_length': MAX_LENGTH, 'best_epoch': best_epoch,
                    'validation_f1': best_f1, 'seed': SEED}, checkpoint_path)
        tokenizer.save_pretrained(OUTPUT_DIR / 'tokenizer')
    else:
        stale_epochs += 1
    (OUTPUT_DIR / 'training_history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
    if stale_epochs >= PATIENCE:
        print(f'Early stopping; best validation F1={best_f1:.4f} at epoch {best_epoch}')
        break

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
test_report = evaluate(model, loaders['test'], device, amp_dtype)
evaluation = {key: value for key, value in test_report.items() if key != 'labels'}
evaluation.update({'best_epoch': best_epoch, 'best_validation_f1': best_f1,
                   'test_video_ids': test_df.video_id.astype(str).tolist(),
                   'test_labels': test_report['labels']})
(OUTPUT_DIR / 'evaluation_results.json').write_text(json.dumps(evaluation, indent=2), encoding='utf-8')
print(json.dumps({key: value for key, value in evaluation.items() if key not in {'probabilities', 'predictions', 'test_video_ids', 'test_labels'}}, indent=2))

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

C:\Users\Harsh\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Harsh\.cache\huggingface\hub\models--distilbert--distilbert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

C:\Users\Harsh\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Harsh\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  263MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'epoch': 1, 'train_loss': 0.35127744023889507, 'validation': {'accuracy': 0.9526813880126183, 'precision': 0.8823529411764706, 'recall': 0.8955223880597015, 'f1': 0.8888888888888888, 'roc_auc': 0.9787164179104478, 'confusion_matrix': [[484, 16], [14, 120]], 'loss': 0.18416200280189515}, 'seconds': 12.59}
{'epoch': 2, 'train_loss': 0.16226690202145963, 'validation': {'accuracy': 0.944794952681388, 'precision': 0.896, 'recall': 0.835820895522388, 'f1': 0.8648648648648649, 'roc_auc': 0.9812537313432836, 'confusion_matrix': [[487, 13], [22, 112]], 'loss': 0.18691383749246598}, 'seconds': 11.82}
{'epoch': 3, 'train_loss': 0.13560454777718797, 'validation': {'accuracy': 0.9542586750788643, 'precision': 0.9069767441860465, 'recall': 0.8731343283582089, 'f1': 0.8897338403041825, 'roc_auc': 0.983455223880597, 'confusion_matrix': [[488, 12], [17, 117]], 'loss': 0.16976219713687896}, 'seconds': 11.8}
{'epoch': 4, 'train_loss': 0.11714816035457408, 'validation': {'accuracy': 0.944794952681388, 'p

## Inference

After training, set `PREDICTION_CSV` to a CSV with `video_id`, `title`, and `description`, then run the next cell. It writes predicted clickbait probabilities and labels to `predictions.csv`.

In [4]:
PREDICTION_CSV = ROOT / 'new_videos.csv'
PREDICTION_OUTPUT = OUTPUT_DIR / 'predictions.csv'

def predict_csv(input_path, output_path=PREDICTION_OUTPUT, batch_size=8):
    frame = pd.read_csv(input_path, dtype={'video_id': 'string'})
    required = {'video_id', 'title', 'description'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'Prediction CSV is missing columns: {sorted(missing)}')
    frame = frame.copy(); frame['title'] = frame.title.fillna('').astype(str)
    frame['description'] = frame.description.map(clean_description)
    if frame.title.str.strip().eq('').any():
        raise ValueError('Prediction input includes an empty title')
    prediction_rows = frame.assign(label=0)
    loader = DataLoader(PairDataset(prediction_rows, tokenizer), batch_size=batch_size, shuffle=False)
    probabilities = []
    model.eval()
    with torch.inference_mode():
        for batch in loader:
            batch = {key: value.to(device) for key, value in batch.items() if key != 'labels'}
            logits = model(**batch)['logits'].float()
            probabilities.extend(logits.softmax(1)[:, 1].cpu().tolist())
    result = frame[['video_id']].copy()
    result['clickbait_probability'] = probabilities
    result['predicted_label'] = (np.asarray(probabilities) >= 0.5).astype(int)
    output_path = Path(output_path); output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False)
    return result

# Uncomment after setting PREDICTION_CSV to an existing input file.
# predictions = predict_csv(PREDICTION_CSV)
# predictions.head()